# Decision Tree — From Scratch
> CART algorithm using Gini impurity. No sklearn.

Recursively splits data by the feature+threshold that minimises impurity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
np.random.seed(42)

## Node & Tree Classes

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None,
                 right=None, value=None):
        self.feature   = feature
        self.threshold = threshold
        self.left      = left
        self.right     = right
        self.value     = value   # set for leaf nodes

class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    # ── Gini impurity ──────────────────────────────────────────────────────
    def _gini(self, y):
        classes, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return 1 - np.sum(p**2)

    # ── Best split ─────────────────────────────────────────────────────────
    def _best_split(self, X, y):
        best_gini, best_feat, best_thr = float('inf'), None, None
        n, d = X.shape
        for feat in range(d):
            thresholds = np.unique(X[:, feat])
            for thr in thresholds:
                left  = y[X[:, feat] <= thr]
                right = y[X[:, feat] >  thr]
                if len(left)==0 or len(right)==0: continue
                g = (len(left)*self._gini(left) + len(right)*self._gini(right)) / n
                if g < best_gini:
                    best_gini, best_feat, best_thr = g, feat, thr
        return best_feat, best_thr

    # ── Build tree ─────────────────────────────────────────────────────────
    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split or len(np.unique(y))==1:
            return Node(value=Counter_mode(y))
        feat, thr = self._best_split(X, y)
        if feat is None:
            return Node(value=Counter_mode(y))
        mask = X[:, feat] <= thr
        return Node(feature=feat, threshold=thr,
                    left =self._build(X[mask],  y[mask],  depth+1),
                    right=self._build(X[~mask], y[~mask], depth+1))

    def fit(self, X, y):
        self.root = self._build(np.array(X), np.array(y), depth=0)

    def _traverse(self, node, x):
        if node.value is not None: return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(node.left,  x)
        return     self._traverse(node.right, x)

    def predict(self, X):
        return np.array([self._traverse(self.root, x) for x in X])

def Counter_mode(arr):
    vals, counts = np.unique(arr, return_counts=True)
    return vals[np.argmax(counts)]

## Train on Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTree(max_depth=5)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")

## Accuracy vs Max Depth

In [ ]:
depths = range(1, 15)
train_accs, test_accs = [], []
for d in depths:
    m = DecisionTree(max_depth=d)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(8,4))
plt.plot(depths, train_accs, 'o-', label='Train')
plt.plot(depths, test_accs,  's-', label='Test')
plt.xlabel("Max Depth"); plt.ylabel("Accuracy")
plt.title("Decision Tree – Depth vs Accuracy")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()